# 01c — Baseline (ALS) & Lissage (Savitzky–Golay)

In [ ]:

import os, sys, numpy as np, matplotlib.pyplot as plt
sys.path.append(os.path.join("..","src"))
from raman_ai.io.standards import load_reference_spectra_mode

def baseline_als(y, lam=1e5, p=0.01, niter=10):
    L = len(y)
    D = np.diff(np.eye(L), 2)
    DTD = D.T @ D
    w = np.ones(L)
    for _ in range(niter):
        W = np.diag(w)
        Z = W + lam * DTD
        b = np.linalg.solve(Z, w * y)
        w = p * (y > b) + (1 - p) * (y <= b)
    return b

def savgol(y, window=5, poly=2):
    y = np.asarray(y, float)
    if window % 2 == 0: window += 1
    half = window // 2
    A = np.vander(np.arange(-half, half+1), N=poly+1, increasing=True)
    ATA = A.T @ A
    inv = np.linalg.pinv(ATA) @ A.T
    coeff = inv[0]
    out = np.copy(y)
    for i in range(half, len(y)-half):
        out[i] = np.dot(coeff, y[i-half:i+half+1])
    out[:half] = y[:half]; out[-half:] = y[-half:]
    return out

root = os.path.join("..","data","standards")
specs, metas = load_reference_spectra_mode(root, mode="synthetic")
s = specs[0]
y = np.asarray(s.intensity, float)
b = baseline_als(y, lam=1e4, p=0.01, niter=10)
ys = savgol(y - b, window=5, poly=2)

import matplotlib.pyplot as plt
plt.figure(); plt.plot(s.wavenumber, y, label="raw"); plt.plot(s.wavenumber, b, label="baseline")
plt.title("ALS baseline"); plt.xlabel("cm^-1"); plt.ylabel("a.u."); plt.tight_layout(); plt.show()

plt.figure(); plt.plot(s.wavenumber, y - b, label="detrended"); plt.plot(s.wavenumber, ys, label="smoothed")
plt.title("Detrended + Savitzky–Golay"); plt.xlabel("cm^-1"); plt.ylabel("a.u."); plt.tight_layout(); plt.show()
